In [1]:
!pip install transformers==4.49.0 qwen_vl_utils accelerate>=0.26.0 PEFT -U

### Upload your cheque image

Please upload the image of your cheque here. After uploading, the code below will process it.

In [2]:
from google.colab import files
from PIL import Image

uploaded = files.upload()

# Assuming only one file is uploaded
image_filename = next(iter(uploaded))
image = Image.open(image_filename)

print(f"Image '{image_filename}' uploaded successfully.")

Saving WhatsApp Image 2026-05-19 at 4.23.34 PM (1).jpeg to WhatsApp Image 2026-05-19 at 4.23.34 PM (1).jpeg
Image 'WhatsApp Image 2026-05-19 at 4.23.34 PM (1).jpeg' uploaded successfully.


In [3]:
!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.8 MB/s eta 0:00:00


In [4]:
from PIL import Image
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
import torch
import os
from qwen_vl_utils import process_vision_info



model_name = "NAMAA-Space/Qari-OCR-v0.3-VL-2B-Instruct"
model = Qwen2VLForConditionalGeneration.from_pretrained(
                model_name,
                torch_dtype="auto",
                device_map="auto"
            )
processor = AutoProcessor.from_pretrained(model_name)
max_tokens = 2000

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/4.42G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/572 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/392 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

In [11]:
import os
from PIL import Image
# Make sure to import your specific model components:
# from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
# from qwen_vl_utils import process_vision_info

# --- SETUP VARIABLES (Make sure these are initialized in your actual environment) ---
# model = Qwen2VLForConditionalGeneration.from_pretrained("Qwen/Qwen2-VL-7B-Instruct", torch_dtype="auto", device_map="auto")
# processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-7B-Instruct")
max_tokens = 512

# 1. Assume you have your PIL Image object loaded
# image = Image.open("your_check.jpg")

src = "/content/WhatsApp Image 2026-05-19 at 4.23.34 PM (1).jpeg"
image.save(src)

# Upgraded prompt to force strict JSON compliance without conversational text
prompt = """You are a strict data extraction engine. Analyze the provided check image.
Extract these exact fields:
- Pay To
- Amount in letters
- Amount in digits
- Date
- MICR

Return ONLY valid JSON in this exact structure.
Do not wrap it in markdown code blocks like ```json.
Do not add any introductory or concluding text.
If a field is missing, leave it as an empty string "".

{
  "Pay To": "",
  "Amount in letters": "",
  "Amount in digits": "",
  "Date": "",
  "MICR": ""
}"""

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": f"file://{src}"},
            {"type": "text", "text": prompt},
        ],
    }
]

# Process and tokenize inputs
text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
image_inputs, video_inputs = process_vision_info(messages)

inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)
inputs = inputs.to("cuda")

# Generate response
generated_ids = model.generate(**inputs, max_new_tokens=max_tokens)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]

output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)[0]

# Clean up local image file safely
if os.path.exists(src):
    os.remove(src)

# Post-processing fix: Sometimes VLMs still wrap responses in markdown fences despite instructions
output_text = output_text.replace("```json", "").replace("```", "").strip()

print(output_text)

{
  "Pay To": "ALEXBANK Intesa Sanpaolo Group",
  "Amount in letters": "مهم شهر سبتمبر",
  "Amount in digits": "2700058",
  "Date": "13-May-26",
  "MICR": "DDMMYYYY"
}
